Importing the data


In [ ]:
import pandas as pd

df_dta = pd.read_stata('Microdata/vup_s0_s1_s2_s3_s4_s6a_s6b_s6c_person.dta')
df_dta.to_csv('nisr_vup_demographics.csv', index=False)

C:\Users\THANKPAD\AppData\Local\Temp\ipykernel_10512\1818310816.py:3: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df = pd.read_stata('Microdata/vup_s0_s1_s2_s3_s4_s6a_s6b_s6c_person.dta')


Filtering the youth.


In [25]:
df = pd.read_csv('nisr_vup_demographics.csv')

#filter for youth 16 to 30years old
df['s1q3y'] = pd.to_numeric(df['s1q3y'], errors='coerce')
df_youth = df[(df['s1q3y'] >= 16) & (df['s1q3y'] <= 30)].copy()

Select the earnings(s6bq1) as a target and dropping youth without listed income
And keeping only features or columns for my model
Function to normalize to monthly income

In [31]:
df_youth['s6bq9a'] = pd.to_numeric(df_youth['s6bq9a'], errors='coerce')
df_clean = df_youth.dropna(subset=['s6bq9a', 's6bq9b']).copy()


def normalize_to_monthly(row):
    amount = row['s6bq9a']
    unit = str(row['s6bq9b']).strip().upper() 
    
    if 'HOUR' in unit: return amount * 8 * 22 
    elif 'DAY' in unit: return amount * 22
    elif 'WEEK' in unit: return amount * 4.33
    elif 'FORTNIGHT...4' in unit: return amount * 2
    elif 'MONTH.' in unit: return amount
    elif 'QUARTER' in unit: return amount / 3
    elif 'YEAR.' in unit: return amount / 12
    else: return None


df_clean['monthly_income'] = df_clean.apply(normalize_to_monthly, axis=1)
df_clean = df_clean.dropna(subset=['monthly_income']).copy()

target = 'monthly_income'

features = ['s1q3y', 's1q2', 'ur', 'province', 'poverty', 's4aq7', 'quintile']

df_final = df_clean[features + [target]].dropna().copy()

print(f"Final usable rows: {df_final.shape[0]}")
df_final.to_csv('done.csv',index=False)

Final usable rows: 1381
